# NexaTel CDR Pipeline — Task 3: Aggregate Usage (Gold)

> **Industry context — NexaTel Communications**  
> After cleansing, CDR records are aggregated into a subscriber-level usage  
> summary. This **Gold table** feeds the monthly invoice generation system  
> and the executive KPI dashboard.  
> This notebook is **Task 3** in the nightly Lakeflow Job.

**Lakeflow Job DAG position:** `ingest_cdr` → `cleanse_cdr` → **`aggregate_usage`**  
**Depends on:** `cleanse_cdr` (run-if condition: **All succeeded**)

In [ ]:
# ── CELL 1: Parameters and upstream task values ───────────────────────────────

dbutils.widgets.text("billing_month",  "2024-12", "Billing Month (YYYY-MM)")
dbutils.widgets.text("source_catalog", "trainer_demo", "Source Catalog")
dbutils.widgets.text("source_schema",  "demo_11",       "Source Schema")

billing_month  = dbutils.widgets.get("billing_month")
source_catalog = dbutils.widgets.get("source_catalog")
source_schema  = dbutils.widgets.get("source_schema")

try:
    silver_count  = int(dbutils.jobs.taskValues.get(taskKey="cleanse_cdr", key="silver_record_count"))
    rejected      = int(dbutils.jobs.taskValues.get(taskKey="cleanse_cdr", key="rejected_count"))
    print(f"Upstream task values — silver_record_count: {silver_count:,}, rejected: {rejected:,}")
except Exception:
    silver_count, rejected = None, None
    print("[Standalone mode] running without upstream task values")

In [ ]:
# ── CELL 2: Read Silver CDR and subscriber profiles ───────────────────────────

from pyspark.sql import functions as F

silver_cdr = spark.read.table(f"{source_catalog}.{source_schema}.silver_cdr")
profiles   = spark.read.table(f"{source_catalog}.{source_schema}.subscriber_profiles")

print(f"silver_cdr records : {silver_cdr.count():,}")
print(f"subscriber profiles: {profiles.count():,}")

In [ ]:
# ── CELL 3: Aggregate usage per subscriber ────────────────────────────────────
# Trainer: explain each metric and why NexaTel's billing system needs it.
#   - total_calls       : needed for per-call pricing plans
#   - total_voice_mins  : billed in 60-second increments
#   - total_data_mb     : overage charges kick in above monthly_cap_gb
#   - total_sms_count   : flat-rate or metered SMS
#   - failed_calls      : FAILED or DROPPED = potential credit claim by customer
#   - roaming_calls     : higher per-minute rate

usage_summary = (
    silver_cdr
    .groupBy("subscriber_id")
    .agg(
        F.count("cdr_id")                                              .alias("total_calls"),
        F.round(
            F.sum(F.when(F.col("call_type") == "voice",
                         F.col("duration_seconds") / 60).otherwise(0)), 2
        )                                                               .alias("total_voice_mins"),
        F.round(F.sum(F.coalesce("data_volume_mb", F.lit(0))), 2)      .alias("total_data_mb"),
        F.sum(F.when(F.col("call_type") == "sms", 1).otherwise(0))    .alias("total_sms_count"),
        F.sum(
            F.when(F.col("termination_code").isin("FAILED", "DROPPED"), 1).otherwise(0)
        )                                                               .alias("failed_calls"),
        F.sum(F.when(F.col("roaming") == True, 1).otherwise(0))       .alias("roaming_calls"),
    )
    .withColumn("_billing_month", F.lit(billing_month))
    .withColumn("_processed_at",  F.current_timestamp())
)

# Enrich with subscriber plan information
gold_usage = (
    usage_summary
    .join(profiles.select("subscriber_id", "plan_type", "monthly_cap_gb", "region"),
          on="subscriber_id", how="left")
    .withColumn(
        "data_overage_mb",
        F.greatest(
            F.col("total_data_mb") - (F.col("monthly_cap_gb") * 1024),
            F.lit(0.0)
        )
    )
)

print(f"Subscribers with usage in {billing_month}: {gold_usage.count():,}")
gold_usage.select(
    "subscriber_id", "plan_type", "total_calls",
    "total_voice_mins", "total_data_mb", "data_overage_mb", "failed_calls"
).orderBy(F.col("total_data_mb").desc()).show(10)

In [ ]:
# ── CELL 4: Write Gold usage summary table ────────────────────────────────────

target_table = f"{source_catalog}.{source_schema}.gold_usage_summary"

(
    gold_usage
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

final_count = gold_usage.count()
print(f"✓ Written {final_count:,} subscriber usage records to {target_table}")

# Reconciliation: compare with upstream task value
if silver_count is not None:
    print(f"\nReconciliation check:")
    print(f"  Silver records processed : {silver_count:,}")
    print(f"  Records read in this task: {silver_cdr.count():,}")
    print(f"  Gold subscribers written : {final_count:,}")

print("\n✓ Task 3 (aggregate_usage) complete.")
dbutils.notebook.exit("SUCCESS")